In [1]:
import numpy as np
from scipy.integrate import solve_ivp, trapezoid
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
%matplotlib widget
plt.close('all')

L = 0.1575 # 35*0.0045 #32 spacers, 4.5mm spacing
# r = 0.005
# stroke = L - 0.0495
L_dead = 0.049
stroke = L - L_dead


mass = 0.088 # just the body, will get overwritten by sweep
mass_foot = 0.0 #launcher
# mass = 0.040
# I_BASELINE = 4e-6
# I = 1.82396e-5  # fitted after correcting measured heights by the 2.5 in origin offset
I = 1.86e-6 # just the rotor
print(f"stroke: {stroke}")

params = {
    'm': mass,
    'I': I,
    'stroke': stroke,
    'g': 9.81,
    'voltage': 6.0,
}

def get_y_tsa(theta):
    val = L**2 - (r*theta)**2
    if val < 1e-12:
        val = 1e-12
    y = L - np.sqrt(val)
    dy_dtheta = (r**2 * theta) / np.sqrt(val)
    d2y_dtheta2 = (r**2 * L**2) / (val)**(3/2)
    return y, dy_dtheta, d2y_dtheta2

VOLTAGE_POINTS = np.arange(1.0, 11.0)
FREE_SPEED_RPM = np.array([
    2049.7002, 4224.7279, 6390.3895, 8368.1081, 10005.7876,
    12022.9659, 14866.4361, 17137.8066, 19872.9528, 21643.4236,
])
SLOPE_0 = -4.70391607e-5  # Nm/RPM
SLOPE_RATE = 4.52823176e-6  # Nm/RPM/V
SLOPE_KNEE_V = 5.352084
TORQUE_CAP_BASELINE_NM = 0.207999  # previous 90 A model
TORQUE_CAP_NM = 0.260  # measured plateau with the ESC fixed at 120 A

free_speed_rpm_1V = 1500

def get_tau(voltage, rpm):
    # if not 1.0 <= voltage <= 10.0:
    #     raise ValueError('voltage must be within the fitted 1-10 V range')
    # free_speed_rpm = np.interp(voltage, VOLTAGE_POINTS, FREE_SPEED_RPM)
    # slope = SLOPE_0 + SLOPE_RATE * min(voltage, SLOPE_KNEE_V)
    # uncapped_tau = slope * (rpm - free_speed_rpm)
    # return min(uncapped_tau, TORQUE_CAP_NM)
    stall_torque_1V = 0.22/12
    return stall_torque_1V * (voltage - rpm/free_speed_rpm_1V)



def integrate_theta(get_y, get_tau, params, theta0=0, theta_dot0=0, fric=0):
    m = params['m']
    I = params['I']
    stroke = params['stroke']
    g = params['g']
   
    def dynamics(t, state):
        theta, theta_dot = state
        y, dy_dtheta, d2y_dtheta2 = get_y(theta)
        rpm = theta_dot * 60 / (2 * np.pi)
        tau = get_tau(params['voltage'], rpm)
        theta_ddot = (-theta_dot**2 * m * dy_dtheta * d2y_dtheta2 - m*g*dy_dtheta - fric*dy_dtheta + tau) / (I + m * (dy_dtheta)**2)
        return [theta_dot, theta_ddot]
    
    def y_target_event(t, state):
        theta, theta_dot = state
        y, _, _ = get_y(theta)
        return y - stroke
    
    y_target_event.terminal = True
    y_target_event.direction = 0

    t_max = 2.0
    t_eval = np.arange(0, t_max, 0.0001)
    
    sol = solve_ivp(
        dynamics,
        t_span=(0, t_max),
        y0=[theta0, theta_dot0],
        method='BDF',
        t_eval=t_eval,
        events=y_target_event,
        dense_output=True,
        rtol=1e-6,
        atol=1e-9,
    )
    
    t = np.append(sol.t, sol.t_events[0][0])
    theta, theta_d = sol.sol(t)
    theta_dd = np.zeros_like(theta)
    
    y = np.zeros_like(theta)
    y_d = np.zeros_like(theta)
    y_dd = np.zeros_like(theta)
    tau = np.zeros_like(theta)
    force = np.zeros_like(theta)
    for i, (th, th_d) in enumerate(zip(theta, theta_d)):
        y_val, dy_dtheta, d2y_dtheta2 = get_y(th)
        _,th_dd = dynamics(t[i], [th, th_d])
        theta_dd[i] = th_dd
        y[i] = y_val
        y_d[i] = dy_dtheta * th_d
        y_dd[i] = dy_dtheta * th_dd + d2y_dtheta2 * th_d**2
        rpm = th_d * 60 / (2 * np.pi)
        tau[i] = get_tau(params['voltage'], rpm)
        force[i] = m * (g + y_dd[i])

    
    return pd.DataFrame({
        't': t,
        'theta': theta,
        'theta_d': theta_d,
        'theta_dd': theta_dd,
        'y': y,
        'y_d': y_d,
        'y_dd': y_dd,
        'tau': tau,
        'force': force,
    })


mass_tests = np.linspace(0.020, 0.220, 20)
radius_tests = [0.005, 0.007, 0.009]
length_tests = [0.154, 0.180]
jump_rows = []

for L in length_tests:
    params['stroke'] = L - L_dead
    for body_mass in mass_tests:
        for r_test in radius_tests:
            params['m'] = body_mass
            r = r_test
            res = integrate_theta(get_y_tsa, get_tau, params, theta0=0, theta_dot0=0)
            final_y_d = res['y_d'].iloc[-1]
            v_takeoff = final_y_d * body_mass / (body_mass + mass_foot)
            jump_height = v_takeoff**2 / (2 * params['g'])
            jump_rows.append({
                'L_m': L,
                'body_mass_kg': body_mass,
                'radius_m': r_test,
                'final_y_d_m_s': final_y_d,
                'v_takeoff_m_s': v_takeoff,
                'jump_height_m': jump_height,
                'energy_J': 0.5 * body_mass * v_takeoff**2,
                'avg_power_W': trapezoid(res['force'] * res['y_d'], res['t']) / res['t'].iloc[-1],
                'avg_motor_power_W': trapezoid(res['tau'] * res['theta_d'], res['t']) / res['t'].iloc[-1],
            })


jump_results = pd.DataFrame(jump_rows)
max_motor_power = (get_tau(params['voltage'], 0) * free_speed_rpm_1V
                   * params['voltage'] * 2 * np.pi / 60 / 4)


radius_colors = dict(zip(radius_tests, plt.cm.viridis(np.linspace(0.08, 0.9, len(radius_tests)))))
length_styles = dict(zip(length_tests, ['--', '-', ':']))
legend = [Line2D([], [], color=color, label=f'r = {1000*r:.0f} mm')
          for r, color in radius_colors.items()]
legend += [Line2D([], [], color='black', linestyle=style, label=f'L = {1000*L:.1f} mm')
           for L, style in length_styles.items()]

for column, ylabel in [('energy_J', 'Takeoff energy (J)'),
                       ('avg_power_W', 'Average body power (W)'),
                       ('avg_motor_power_W', 'Average motor power (W)')]:
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for (L, r), rows in jump_results.groupby(['L_m', 'radius_m']):
        ax.plot(rows['body_mass_kg'], rows[column], color=radius_colors[r],
                linestyle=length_styles[L], linewidth=2)
    plot_legend = legend.copy()
    if column in ['avg_motor_power_W', 'avg_power_W']:
        plot_legend.append(ax.axhline(max_motor_power, color='black', linestyle='-.',  label=f'Max motor power: {max_motor_power:.2f} W'))
    ax.set(xlabel='Body mass (kg)', ylabel=ylabel, ylim=(0, None),
           title=f'{params["voltage"]:.0f} V, foot mass {1000 * mass_foot:.0f} g')
    ax.grid(alpha=0.3)
    ax.legend(handles=plot_legend, fontsize=8)
    fig.tight_layout()
    fig.savefig(f'results/{column}.png', dpi=180)


stroke: 0.1085


In [2]:
L, r = length_tests[0], radius_tests[0]
first_params = params | {'m': mass_tests[0], 'stroke': L - L_dead}
first = integrate_theta(get_y_tsa, get_tau, first_params)
t = first['t'] * 1000
body_power = first['force'] * first['y_d']
rotor_power = first_params['I'] * first['theta_d'] * first['theta_dd']
shaft_power = first['tau'] * first['theta_d']

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.stackplot(t, body_power, rotor_power.clip(lower=0), alpha=0.6,
             labels=['Body (kinetic + gravity)', 'Rotor (+ stored, − released)'])
ax.fill_between(t, rotor_power.clip(upper=0), 0, color='C1', alpha=0.6)
ax.plot(t, shaft_power, color='black', label='Motor shaft')
ax.axhline(0, color='gray', linewidth=0.5)
ax.set(xlabel='Time (ms)', ylabel='Power (W)',
       title=f'{1000*first_params["m"]:.0f} g, r = {1000*r:.0f} mm, L = {1000*L:.1f} mm')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()

fig.savefig('results/power_distribution.png', dpi=180)


In [3]:
jump_results.to_csv('results/length_sweep.csv', index=False)
assert len(jump_results) == 120
assert np.isfinite(jump_results.select_dtypes(include='number')).all().all()
assert jump_results['avg_motor_power_W'].le(max_motor_power * 1.001).all()
